In [3]:
"""
2D Truss Analyzer
=================
Solves statically determinate 2D pin-jointed trusses using:
  1. Method of Joints   -> global matrix / linear algebra approach
                            (solves ALL member forces + reactions at once)
  2. Method of Sections -> equilibrium of an isolated sub-structure
                            (solves up to 3 specific member forces directly,
                            the classic hand-calc shortcut)

Includes matplotlib visualization of truss geometry, supports, applied
loads, and solved member forces (tension in red, compression in blue).

Sign convention: positive member force = TENSION, negative = COMPRESSION.

Dependencies:
    pip install numpy matplotlib
"""

from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches


# --------------------------------------------------------------------------
# Data structures
# --------------------------------------------------------------------------

@dataclass
class Node:
    id: int
    x: float
    y: float
    # "free", "pin", "roller_x" (restrains horizontal motion),
    # "roller_y" (restrains vertical motion -- the common roller-on-ground case)
    support: str = "free"

    def reaction_dofs(self) -> List[str]:
        if self.support == "pin":
            return [f"Rx{self.id}", f"Ry{self.id}"]
        elif self.support == "roller_x":
            return [f"Rx{self.id}"]
        elif self.support == "roller_y":
            return [f"Ry{self.id}"]
        return []


@dataclass
class Member:
    id: int
    ni: int
    nj: int


@dataclass
class Load:
    node: int
    fx: float = 0.0
    fy: float = 0.0


# --------------------------------------------------------------------------
# Truss solver
# --------------------------------------------------------------------------

class Truss:
    def __init__(self, nodes: List[Node], members: List[Member], loads: Optional[List[Load]] = None):
        self.nodes = {n.id: n for n in nodes}
        self.members = members
        self.loads = loads or []

    # ---------------- geometry helpers ----------------
    def member_vector(self, m: Member) -> Tuple[float, float, float]:
        ni, nj = self.nodes[m.ni], self.nodes[m.nj]
        dx, dy = nj.x - ni.x, nj.y - ni.y
        L = float(np.hypot(dx, dy))
        return dx / L, dy / L, L

    def all_reaction_dofs(self) -> List[str]:
        dofs = []
        for n in self.nodes.values():
            dofs.extend(n.reaction_dofs())
        return dofs

    def determinacy(self) -> str:
        j = len(self.nodes)
        m = len(self.members)
        r = len(self.all_reaction_dofs())
        lhs, rhs = m + r, 2 * j
        status = ("statically determinate" if lhs == rhs else
                  "statically indeterminate" if lhs > rhs else
                  "unstable (mechanism)")
        return f"m={m}, r={r}, j={j}  ->  m+r={lhs} vs 2j={rhs}  =>  {status}"

    # ---------------- Method of joints (full matrix solve) ----------------
    def solve_joints(self) -> Dict[str, float]:
        """
        Assemble 2 equilibrium equations (Fx, Fy) per joint and solve the
        resulting linear system for every member force and every reaction
        component simultaneously.
        """
        node_ids = sorted(self.nodes.keys())
        idx = {nid: i for i, nid in enumerate(node_ids)}
        n_eq = 2 * len(node_ids)

        reaction_dofs = self.all_reaction_dofs()
        unknowns = [("m", m.id) for m in self.members] + [("r", d) for d in reaction_dofs]
        n_unk = len(unknowns)

        if n_unk != n_eq:
            raise ValueError(
                f"System is not exactly determinate ({n_unk} unknowns vs {n_eq} equations).\n"
                f"Determinacy check: {self.determinacy()}"
            )

        A = np.zeros((n_eq, n_unk))
        b = np.zeros(n_eq)

        for ld in self.loads:
            r0 = 2 * idx[ld.node]
            b[r0] -= ld.fx
            b[r0 + 1] -= ld.fy

        for col, m in enumerate(self.members):
            dx, dy, _ = self.member_vector(m)
            i_row, j_row = 2 * idx[m.ni], 2 * idx[m.nj]
            A[i_row, col] += dx      # tension pulls node i TOWARD node j
            A[i_row + 1, col] += dy
            A[j_row, col] += -dx     # and pulls node j TOWARD node i
            A[j_row + 1, col] += -dy

        col0 = len(self.members)
        for col, dof in enumerate(reaction_dofs, start=col0):
            nid = int(dof[2:])
            row = 2 * idx[nid]
            if dof.startswith("Rx"):
                A[row, col] = 1.0
            else:
                A[row + 1, col] = 1.0

        x = np.linalg.solve(A, b)

        result = {}
        for (kind, key), val in zip(unknowns, x):
            if kind == "m":
                result[f"Member {key}"] = val
            else:
                result[key] = val
        self.last_solution = result
        return result

    # ---------------- Global reactions only (3-equation shortcut) --------
    def solve_reactions_global(self) -> Dict[str, float]:
        """
        Solve reactions from overall equilibrium (Fx=0, Fy=0, M=0 about the
        origin). Requires exactly 3 unknown reaction components -- the
        normal case for a determinate truss with one pin + one roller.
        """
        reaction_dofs = self.all_reaction_dofs()
        if len(reaction_dofs) != 3:
            raise ValueError("Global 3-equation method needs exactly 3 reaction unknowns; "
                              "use solve_joints() for other support layouts.")

        A = np.zeros((3, 3))
        b = np.zeros(3)

        for ld in self.loads:
            n = self.nodes[ld.node]
            b[0] -= ld.fx
            b[1] -= ld.fy
            b[2] -= (n.x * ld.fy - n.y * ld.fx)

        for col, dof in enumerate(reaction_dofs):
            nid = int(dof[2:])
            n = self.nodes[nid]
            if dof.startswith("Rx"):
                A[0, col] = 1.0
                A[2, col] = -n.y
            else:
                A[1, col] = 1.0
                A[2, col] = n.x

        x = np.linalg.solve(A, b)
        return dict(zip(reaction_dofs, x))

    # ---------------- Method of sections ----------------
    def method_of_sections(self, cut_member_ids: List[int], sub_node_ids: List[int],
                            moment_point: Optional[Tuple[float, float]] = None,
                            reactions: Optional[Dict[str, float]] = None) -> Dict[int, float]:
        """
        Cut up to 3 members, isolate the free body made of `sub_node_ids`,
        and solve the 3 equilibrium equations of that free body
        (Fx=0, Fy=0, M about `moment_point`=0) for the cut member forces.
        Each cut member is assumed in tension; a negative result means
        compression -- exactly like the hand-calculation method.
        """
        if len(cut_member_ids) > 3:
            raise ValueError("A section cut must pass through 3 members or fewer.")
        if reactions is None:
            reactions = self.solve_reactions_global()

        sub_nodes = set(sub_node_ids)
        cut_members = [m for m in self.members if m.id in cut_member_ids]

        if moment_point is None:
            first = self.nodes[sub_node_ids[0]]
            moment_point = (first.x, first.y)
        px, py = moment_point

        Fx_known = Fy_known = M_known = 0.0

        for ld in self.loads:
            if ld.node in sub_nodes:
                n = self.nodes[ld.node]
                Fx_known += ld.fx
                Fy_known += ld.fy
                M_known += (n.x - px) * ld.fy - (n.y - py) * ld.fx

        for n in self.nodes.values():
            if n.id in sub_nodes:
                for dof in n.reaction_dofs():
                    if dof in reactions:
                        val = reactions[dof]
                        if dof.startswith("Rx"):
                            Fx_known += val
                            M_known += -(n.y - py) * val
                        else:
                            Fy_known += val
                            M_known += (n.x - px) * val

        A = np.zeros((3, len(cut_members)))
        for col, m in enumerate(cut_members):
            ni, nj = self.nodes[m.ni], self.nodes[m.nj]
            dx, dy, _ = self.member_vector(m)
            if m.ni in sub_nodes:
                ax, ay = ni.x, ni.y
                ux, uy = dx, dy
            else:
                ax, ay = nj.x, nj.y
                ux, uy = -dx, -dy
            A[0, col] = ux
            A[1, col] = uy
            A[2, col] = (ax - px) * uy - (ay - py) * ux

        b = -np.array([Fx_known, Fy_known, M_known])

        if len(cut_members) == 3:
            x = np.linalg.solve(A, b)
        else:
            x, *_ = np.linalg.lstsq(A, b, rcond=None)

        return {m.id: float(val) for m, val in zip(cut_members, x)}

    # ---------------- Visualization ----------------
    def plot(self, forces: Optional[Dict[str, float]] = None, title="2D Truss",
             ax=None, show=True, savepath=None):
        created = False
        if ax is None:
            fig, ax = plt.subplots(figsize=(8, 6))
            created = True

        for m in self.members:
            ni, nj = self.nodes[m.ni], self.nodes[m.nj]
            color, lw, label_val = "black", 2, None
            if forces is not None:
                f = forces.get(f"Member {m.id}", forces.get(m.id))
                if f is not None:
                    color = "crimson" if f > 1e-6 else ("steelblue" if f < -1e-6 else "gray")
                    lw = 1.5 + min(abs(f), 10) / 10 * 2.5
                    label_val = f
            ax.plot([ni.x, nj.x], [ni.y, nj.y], color=color, lw=lw, zorder=1)
            if label_val is not None:
                mx, my = (ni.x + nj.x) / 2, (ni.y + nj.y) / 2
                tag = "T" if label_val > 0 else "C"
                ax.annotate(f"{abs(label_val):.2f} {tag}", (mx, my), fontsize=8,
                            ha="center", va="center",
                            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))

        for n in self.nodes.values():
            ax.plot(n.x, n.y, "ko", ms=5, zorder=3)
            ax.annotate(str(n.id), (n.x, n.y), textcoords="offset points",
                        xytext=(6, 6), fontsize=9, zorder=4)
            if n.support == "pin":
                ax.add_patch(patches.RegularPolygon((n.x, n.y - 0.25), numVertices=3,
                             radius=0.25, orientation=0, color="seagreen", zorder=2))
            elif n.support in ("roller_x", "roller_y"):
                ax.add_patch(patches.Circle((n.x, n.y - 0.22), 0.15, color="darkorange", zorder=2))

        for ld in self.loads:
            n = self.nodes[ld.node]
            ax.annotate("", xy=(n.x + ld.fx * 0.02, n.y + ld.fy * 0.02),
                        xytext=(n.x, n.y),
                        arrowprops=dict(arrowstyle="->", color="purple", lw=2))
            ax.annotate(f"({ld.fx:g},{ld.fy:g})", (n.x + ld.fx * 0.02, n.y + ld.fy * 0.02),
                        color="purple", fontsize=8)

        ax.set_aspect("equal")
        ax.set_title(title)
        ax.grid(alpha=0.3)
        if created:
            if savepath:
                plt.savefig(savepath, dpi=150, bbox_inches="tight")
            if show:
                plt.show()
            plt.close()
        return ax


# --------------------------------------------------------------------------
# Demo
# --------------------------------------------------------------------------

if __name__ == "__main__":
    # Simple 4-node, 5-member truss:
    #
    #        4 (2,2)
    #       / \
    #      /   \
    #     1-----3-----2
    #  (0,0) (2,0)  (4,0)
    #  pin          roller_y
    #
    # 10 kN downward load applied at the apex (node 4)

    nodes = [
        Node(1, 0, 0, support="pin"),
        Node(2, 4, 0, support="roller_y"),
        Node(3, 2, 0, support="free"),
        Node(4, 2, 2, support="free"),
    ]
    members = [
        Member(1, 1, 3),   # bottom-left
        Member(2, 3, 2),   # bottom-right
        Member(3, 1, 4),   # left diagonal
        Member(4, 4, 2),   # right diagonal
        Member(5, 3, 4),   # vertical
    ]
    loads = [Load(node=4, fx=0, fy=-10)]

    truss = Truss(nodes, members, loads)

    print(truss.determinacy())
    print()

    print("=== Method of Joints (full matrix solve) ===")
    joint_forces = truss.solve_joints()
    for k, v in joint_forces.items():
        print(f"  {k:12s}: {v:8.3f}")
    print()

    print("=== Method of Sections (cutting members 2, 3, 5 -> isolating {1,3}) ===")
    reactions = truss.solve_reactions_global()
    section_forces = truss.method_of_sections(
        cut_member_ids=[2, 3, 5],
        sub_node_ids=[1, 3],
        reactions=reactions,
    )
    for mid, val in section_forces.items():
        print(f"  Member {mid}: {val:8.3f}   (Method of Joints gave {joint_forces[f'Member {mid}']:8.3f})")
    print()

    truss.plot(forces=joint_forces, title="Truss - Member Forces (T=tension, C=compression)",
               show=False, savepath="/home/rhysowens/projects/statics-asen2401/truss_demo.png")
    print("Saved plot to truss_demo.png")

m=5, r=3, j=4  ->  m+r=8 vs 2j=8  =>  statically determinate

=== Method of Joints (full matrix solve) ===
  Member 1    :    5.000
  Member 2    :    5.000
  Member 3    :   -7.071
  Member 4    :   -7.071
  Member 5    :    0.000
  Rx1         :    0.000
  Ry1         :    5.000
  Ry2         :    5.000

=== Method of Sections (cutting members 2, 3, 5 -> isolating {1,3}) ===
  Member 2:    5.000   (Method of Joints gave    5.000)
  Member 3:   -7.071   (Method of Joints gave   -7.071)
  Member 5:    0.000   (Method of Joints gave    0.000)

Saved plot to truss_demo.png
